# Phase 1 — Building the Model Input (the 60-day windows)

We turn the cleaned daily returns into the exact form the fingerprinting model needs: lots of short, comparable 60-day snippets, one batch per company, all put on the same scale.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("data/model_input", exist_ok=True)

log_returns = pd.read_parquet("data/processed/log_returns_clean.parquet")

WINDOW_LENGTH = 60   # ~3 months of trading days
STRIDE = 60          # non-overlapping windows

print(f"Log returns: {log_returns.shape}")
print(f"Window: {WINDOW_LENGTH} days, stride: {STRIDE}")
print(f"Expected windows per ticker: {(log_returns.shape[0] - WINDOW_LENGTH) // STRIDE + 1}")

Log returns: (2838, 462)
Window: 60 days, stride: 60
Expected windows per ticker: 47


In [2]:
# Build 3D tensor: (num_assets, num_windows, window_length)
tickers = log_returns.columns.tolist()
returns_matrix = log_returns.values  # (days, tickers)

window_starts = list(range(0, returns_matrix.shape[0] - WINDOW_LENGTH + 1, STRIDE))
num_windows = len(window_starts)
num_assets = len(tickers)

tensor = np.zeros((num_assets, num_windows, WINDOW_LENGTH))

for i, start in enumerate(window_starts):
    tensor[:, i, :] = returns_matrix[start:start + WINDOW_LENGTH, :].T

print(f"Raw tensor shape: {tensor.shape}  (assets x windows x days)")

Raw tensor shape: (462, 47, 60)  (assets x windows x days)


In [3]:
# Z-score normalisation per window (each asset-window independently)
means = tensor.mean(axis=2, keepdims=True)
stds = tensor.std(axis=2, keepdims=True)
stds[stds < 1e-8] = 1.0  # avoid division by zero for flat windows

tensor_norm = (tensor - means) / stds

print(f"Normalised tensor shape: {tensor_norm.shape}")
print(f"Sample window mean: {tensor_norm[0, 0].mean():.6f} (should be ~0)")
print(f"Sample window std:  {tensor_norm[0, 0].std():.6f} (should be ~1)")

Normalised tensor shape: (462, 47, 60)
Sample window mean: 0.000000 (should be ~0)
Sample window std:  1.000000 (should be ~1)


In [4]:
# Save tensor + metadata
np.save("data/model_input/tensor_norm.npy", tensor_norm)
np.save("data/model_input/tensor_raw.npy", tensor)
pd.Series(tickers).to_csv("data/model_input/tickers.csv", index=False, header=False)

# Save window date ranges for traceability
dates = log_returns.index
window_info = pd.DataFrame({
    "window": range(num_windows),
    "start_date": [dates[s].date() for s in window_starts],
    "end_date": [dates[s + WINDOW_LENGTH - 1].date() for s in window_starts]
})
window_info.to_csv("data/model_input/window_dates.csv", index=False)

print(f"Saved tensor_norm.npy: {tensor_norm.shape}")
print(f"Saved tensor_raw.npy: {tensor.shape}")
print(f"Saved tickers.csv: {len(tickers)} tickers")
print(f"Saved window_dates.csv: {num_windows} windows")

Saved tensor_norm.npy: (462, 47, 60)
Saved tensor_raw.npy: (462, 47, 60)
Saved tickers.csv: 462 tickers
Saved window_dates.csv: 47 windows


## Sanity check on the windows

We confirm there are no broken values and that every snippet really is on the same scale, then plot a few.

**How to read the charts:** each line is one company's 60-day snippet of daily moves, after rescaling. They should look like jagged noise centred on zero with a similar overall size — that is the point of the rescaling: the model then compares the *shape* of the wiggles, not whether one stock is simply pricier or jumpier than another.

In [ ]:
print(f'NaN in tensor: {np.isnan(tensor_norm).sum()} | Inf: {np.isinf(tensor_norm).sum()}')

rng = np.random.default_rng(0)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for a, w in zip(rng.integers(0, num_assets, 6), rng.integers(0, num_windows, 6)):
    ax[0].plot(tensor_norm[a, w], lw=0.8, alpha=0.85, label=f'{tickers[a]} w{w}')
ax[0].axhline(0, color='k', lw=0.5); ax[0].legend(fontsize=7)
ax[0].set_title('Six random normalised snippets — same scale, different shapes')
ax[0].set_xlabel('day within the 60-day window'); ax[0].set_ylabel('z-scored daily return')
ax[1].hist(tensor_norm.reshape(-1), bins=100, color='steelblue', density=True)
ax[1].set_title('All normalised returns — bell-shaped, centred on zero, fat tails')
ax[1].set_xlabel('z-scored daily return')
plt.tight_layout(); plt.show()

## Summary

**Input:** the cleaned daily returns (2,838 days × 462 companies).

**The choices we made:**
- **Snippet length: 60 trading days (~3 months).** Long enough to show a medium-term pattern, short enough to give many snippets per company.
- **Non-overlapping snippets.** Each 60-day block is separate from the next, so no day is counted twice and snippets do not bleed into each other.
- **Rescale each snippet to the same footing (average 0, spread 1).** This strips out "how big or pricey" a stock is, so the model focuses purely on the *shape* of its ups and downs.

**Output:** `data/model_input/tensor_norm.npy` — a stack of shape (462 companies × 47 snippets × 60 days).

Phase 2 will feed these snippets to the fingerprinting model — but, importantly, it trains only on the earliest snippets (2015–2017) and then leaves the model frozen, so the later years stay untouched for an honest test.